# 🗂️ 2. 엔터프라이즈 데이터 인덱싱: 정규화 우선 파이프라인과 Graph RAG

1교시의 에이전트가 사용한 검색 도구 뒤에는 **인덱싱된 데이터베이스**가 있습니다.
에이전트 루프는 데이터 품질을 보완해 주지 않습니다. 청크가 맥락을 잃었거나 관계가 끊어져 있으면 어떤 에이전트도 정답을 찾지 못합니다.
실무에서 RAG 품질의 병목은 대부분 모델이 아니라 **문서 파싱과 인덱싱**에 있습니다.

---

### 🎯 학습 목표
1. 인덱싱의 두 가지 핵심 문제(**탈맥락화, 포맷 파편화**)와 해법(**정규화 우선**)을 이해합니다.
2. 한국은행 보고서 PDF를 표준 Markdown으로 정규화하고, 문맥 헤더와 메타데이터를 붙여 **형식과 상관없이 하나의 함수로** 인덱싱합니다.
3. 하이브리드 검색(Dense + BM25)과 윈도우 확장으로 **약어, 조항 번호, 표**를 놓치지 않는 검색기를 만듭니다.
4. 벡터 검색이 못 푸는 **다중 홉 관계 질문**을 지식 그래프와 Microsoft GraphRAG로 해결합니다.

### 🎓 학습 목차

| 파트 | 주제 | 핵심 기법 |
|:---:|:---|:---|
| **Step 0** | 환경 세팅 | |
| **Part 1** | 인덱싱의 두 가지 문제 | 탈맥락화, 포맷 파편화 → 정규화 우선 |
| **Part 2** | 정규화: PDF를 표준 Markdown으로 | PDF 북마크 헤더 승격 (BOK) |
| **Part 3** | 청킹: 청크가 스스로 출처를 말하게 | 문맥 헤더 주입 (TOC, Breadcrumb) |
| **Part 4** | 하나의 함수로 통합 인덱싱 | `index_documents()`, 분기 메타데이터 태깅 |
| **Part 5** | 검색: 의미와 키워드를 함께 | 사내 약어 사전, BM25 + Dense 하이브리드 (RRF) |
| **Part 6** | 생성용 컨텍스트: 작게 찾고 크게 읽기 | Window expansion |
| **Part 7** | 지식 그래프 (직접 구축) | 트리플렛 추출, NetworkX, 다중 홉 탐색 |
| **Part 8** | Microsoft GraphRAG | 커뮤니티 리포트, Global vs Local 검색 |
| **Part 9** | 정리와 Mission 02 | `app/database/` 빌드 및 검증 |

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`)와 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 프로젝트 루트 경로 자동 설정 (app/ 폴더가 있는 위치까지 상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. 환경변수 로드 (프로젝트 루트의 .env)
load_dotenv(os.path.join(project_root, ".env"), override=True)

print(f"✅ Working Directory: {os.getcwd()}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. 공통 유틸리티 로드
from app.utils import init_chat_model, normalize_content

llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)

---
## 🧩 Part 1. 인덱싱의 두 가지 문제

### 문제 1. 탈맥락화 (Decontextualization)

문서를 청크로 자르는 순간, 각 청크는 **자신이 어느 문서의 어느 위치에 있었는지** 잃어버립니다.

```text
원본 문서                                         잘린 청크 (검색 대상)
# 국내 여비 및 교통비 지급 규정
## 제3장 숙박비, 일비 및 식비 지급 기준
### 제15조 (일비 및 식비 정액 지급 기준)
| 구분 | 임원 | 팀장 | 팀원 |                      ┌──────────────────────────────┐
| 일비 | 40,000원 | 30,000원 | 25,000원 |    ──▶   │ 2. 당일 출장의 경우 식비는       │
2. 당일 출장의 경우 식비는 1식(12,000원)을        │ 1식(12,000원)을 지급하며, 일비는 │
   지급하며, 일비는 기준액의 50%를 지급한다.        │ 기준액의 50%를 지급한다.         │
                                                └──────────────────────────────┘
                                                 ❓ 어느 회사, 어떤 규정의 몇 조?
                                                 ❓ "기준액"은 얼마? (표는 앞 청크에 있음)
```

한국은행 보고서라면 더 심각합니다. 같은 문장 `"수출은 전년동기대비 41.4% 증가하였다"`가 **8개 산업 × 4개 분기**마다 반복되므로,
청크만 봐서는 어느 산업, 어느 분기의 수치인지 구분할 수 없습니다.

### 문제 2. 포맷 파편화

사내 규정은 Markdown, 산업 보고서는 다단 편집 PDF, 조직 정보는 일반 텍스트입니다. 형식마다 파이프라인을 따로 만들면 문서 종류만큼 코드가 늘어납니다.

```mermaid
flowchart LR
    subgraph A["❌ 형식마다 파이프라인"]
        a1[".md"] --> p1["MD 청킹기"] --> s1["검색기 A"]
        a2[".pdf"] --> p2["PDF 청킹기"] --> s2["검색기 B"]
        a3[".docx / .hwp"] --> p3["또 다른 청킹기"] --> s3["검색기 C"]
    end
    subgraph B["✅ 정규화 우선"]
        b1[".md"] --> n["정규화기<br/>(형식별로 얇게)"]
        b2[".pdf"] --> n
        b3[".docx / .hwp"] --> n
        n --> m["표준 Markdown<br/>H1~H4 계층"] --> c["청킹기 1개"] --> r["검색기 1개"]
    end
```

### 해법: 정규화 우선 (Normalize First)

**모든 문서를 H1~H4 계층을 가진 표준 Markdown으로 먼저 바꿉니다.** 두 문제를 모두 이 전처리 단계에서 해결합니다.

| 문제 | 전처리가 해결하는 방식 |
|:---|:---|
| **포맷 파편화** | 형식이 무엇이든 같은 규격(표준 Markdown)으로 바꾸므로, 그 뒤의 청킹·인덱싱·검색 코드는 **하나를 공용으로** 씁니다 |
| **탈맥락화** | 표준 Markdown에 계층(문서 > 장 > 조항)이 생기므로, 그 계층을 **모든 청크에 문맥 헤더로** 붙일 수 있습니다 |

새 형식이 들어오면 정규화기 하나만 추가합니다. 같은 유형의 문서(같은 기관이 같은 양식으로 매 분기 내는 보고서 등)는 **설정 하나로 계속 처리**되며,
그 정규화 설정은 AI 코딩 에이전트의 도움을 받아 빠르게 만들 수 있습니다 (Part 2 마지막 참고).

```mermaid
flowchart LR
    MD[".md 사내 규정"] --> N
    PDF[".pdf 한국은행 보고서"] --> N
    N["① 정규화<br/>표준 Markdown"] --> C["② 청킹<br/>문맥 헤더 · 분기 메타데이터"]
    C --> I["③ 인덱스<br/>Dense(Chroma) + BM25"]
    I --> R["④ 검색<br/>약어 사전 → RRF → 윈도우 확장"]
```

### 🗺️ 이 노트북의 기법 지도

코드는 노트북에 펼치지 않고 모듈로 정리했습니다. 동작이 궁금하면 아래 위치를 열어 보세요. (링크를 누르면 파일이 열립니다)

| 단계 | 기법 | 해결하는 문제 | 구현 위치 |
|:---:|:---|:---|:---|
| ① | PDF 북마크 헤더 승격 | 포맷 파편화 (PDF에 계층이 없음) | [`rag/preprocessing/normalize.py` · `pdf_to_markdown()` L12–80](../rag/preprocessing/normalize.py)<br/>[`rag/preprocessing/bok.py` · `BOK_PDF_OPTIONS` L12](../rag/preprocessing/bok.py) |
| ① | 정규화 결과 검수 (텍스트 보존율) | 파서가 조용히 버린 텍스트 | [`rag/preprocessing/quality.py` · `measure_text_coverage()` L8–32](../rag/preprocessing/quality.py) |
| ① | 형식별 정규화기 선택 | 포맷 파편화 | [`rag/preprocessing/normalize.py` · `to_markdown()` L83–91](../rag/preprocessing/normalize.py) |
| ② | 문맥 헤더 주입 (TOC, Breadcrumb) | 탈맥락화 | [`rag/indexing/contextual_headers.py` · `ContextualHeaderEnricher.process_file()` L39–111](../rag/indexing/contextual_headers.py) |
| ② | 분기 메타데이터 태깅 | 같은 문장이 분기마다 반복 | [`rag/preprocessing/bok.py` · `bok_metadata()` L28–34](../rag/preprocessing/bok.py) |
| ①② | 통합 진입점 | 형식별 코드 분기 제거 | [`rag/indexing/pipeline.py` · `index_documents()` L19–59](../rag/indexing/pipeline.py) |
| ④ | 사내 약어 사전 | 질문 용어 ≠ 문서 용어 | [`rag/retrieval/domain_dictionary.py` · `DomainDictionary` L12–65](../rag/retrieval/domain_dictionary.py) |
| ④ | BM25 + Dense 하이브리드 (RRF) | 고유명사·조항 번호 누락 | [`rag/retrieval/hybrid_retriever.py` · `HybridRetriever.search()` L55–84](../rag/retrieval/hybrid_retriever.py)<br/>[`rag/retrieval/hybrid_retriever.py` · `HybridRetriever._reciprocal_rank_fusion()` L96–119](../rag/retrieval/hybrid_retriever.py) |
| ④ | Window expansion | 청크 경계에서 표·단서 조항 잘림 | [`rag/retrieval/context_window_expander.py` · `ChunkWindowExpander.expand_documents()` L25–79](../rag/retrieval/context_window_expander.py) |

---
## 📑 Part 2. 정규화: 모든 문서를 같은 규격의 Markdown으로

### 정규화의 목표: 표준 Markdown 규격

정규화는 "PDF를 텍스트로 바꾸는 일"이 아니라 **모든 문서를 하나의 약속된 규격으로 맞추는 일**입니다. 이 과정의 규격은 다음과 같습니다.

| 규격 요소 | 의미 | 사내 규정 (.md) | 한국은행 보고서 (.pdf) |
|:---|:---|:---|:---|
| **H1** | 문서 안의 최상위 주제 | 규정명 | 산업 (반도체, 자동차, …) |
| **H2~H4** | 장 · 절 · 조항 | 제3장 > 제15조 | 최근 동향 / 향후 전망 |
| **본문** | 헤더 아래의 내용, 표는 Markdown 표 | 조항 본문 | 분석 본문 |
| **메타데이터** | 계층으로 표현되지 않는 속성 | 출처 파일 | 출처 파일, **연도·분기** |

이 규격만 지키면 Part 3 이후의 청킹, 문맥 헤더, 인덱싱, 검색은 **문서가 원래 무엇이었는지 알 필요가 없습니다.**

```mermaid
flowchart LR
    subgraph N["포맷별 정규화기"]
        a[".md 사내 규정<br/>(기존 규격 준수)"]
        b[".pdf 한국은행 보고서<br/>(BOK 전용 정규화 설정)"]
        c[".pdf / .docx 다른 계열<br/>(신규 정규화 설정)"]
    end
    a --> S["표준 Markdown 규격<br/>(H1~H4 계층)"]
    b --> S
    c --> S
    S --> P["공용 파이프라인<br/>(청킹 · 인덱스 · 검색)"]
```

---

### 2.1 문서 1. 사내 규정 (.md): 이미 갖춰진 마크다운 계층 구조 확인

우리가 다루는 첫 번째 문서 계열은 사내 규정입니다.
사내 규정 문서는 이미 텍스트 기반 Markdown 파일이므로, 최상위 규정명(`#`), 장(`##`), 조항(`###`)의 계층 구조를 **명확히 유지**하고 있습니다.
우리가 목표로 하는 **'표준 Markdown 계층 규격'**의 기준점입니다.

In [ ]:
import glob
import os

policy_files = sorted(glob.glob("data/policies/*.md"))
print(f"📂 발견된 사내 규정 파일 ({len(policy_files)}개):")
for f in policy_files:
    print(" -", os.path.basename(f))

# 첫 번째 규정(여비지급규정) 원문의 헤더 계층 확인
with open(policy_files[0], "r", encoding="utf-8") as f:
    sample_policy_text = f.read()

headers = [ln for ln in sample_policy_text.splitlines() if ln.startswith("#")]
print(f"\n📌 {os.path.basename(policy_files[0])}의 마크다운 계층 구조:")
for h in headers[:8]:
    print(" ", h)

💡 사내 규정은 별도의 복잡한 파서 없이도 이미 `# 규정명` ➜ `## 장` ➜ `### 조항`으로 깔끔하게 계층화되어 있습니다.

---

### 2.2 문서 2. 한국은행 산업보고서 (.pdf): 비정형 PDF의 정규화와 헤더 승격

사내 규정과 달리, 실무에서 마주하는 대다수의 문서는 다단 편집과 표로 복잡하게 얽힌 **비정형 PDF 문서**입니다.
PDF 안에는 "이 글자가 제목"이라는 정보가 없습니다. 글자 크기와 위치뿐입니다. 그래서 일반 변환기는 제목을 **추측**하고, 그 과정에서 계층을 잘못 만들거나 일부 텍스트를 **조용히 빠뜨립니다.**

해법은 문서가 **이미 가지고 있는 구조 정보**를 찾아 규격에 대응시키는 것입니다. 한국은행 보고서라면 이렇습니다.

| 문서가 가진 구조 정보 | 규격에서의 위치 |
|:---|:---|
| PDF 전자 북마크(목차)의 산업 항목과 시작 페이지 | **H1** (해당 페이지 맨 위에 `# 산업명` 삽입, 헤더 승격) |
| 산업마다 반복되는 소제목 `최근 동향` / `향후 전망` | **H2** |
| 파일명 `2024_3사분기_…` | **메타데이터** (연도·분기) |
| 매 페이지 상단의 러닝 헤더 | 삭제 (노이즈) |

변환 함수 [`rag/preprocessing/normalize.py` · `pdf_to_markdown()` L12–80](../rag/preprocessing/normalize.py)는 범용이고, 이 보고서 계열에만 해당하는 규칙은 설정 [`rag/preprocessing/bok.py` · `BOK_PDF_OPTIONS` L12](../rag/preprocessing/bok.py)에 모여 있습니다.
설정 하나로 4개 분기 보고서를 모두 처리하며, 다른 계열의 문서가 오면 **함수는 그대로 두고 설정만 새로 씁니다.**

먼저 북마크를 확인합니다.

In [ ]:
import pymupdf

bok_pdf = sorted(glob.glob("data/bok_major_industry_reports/*.pdf"))[0]
doc = pymupdf.open(bok_pdf)
print(f"📄 {bok_pdf} ({len(doc)}페이지)\n")
print("=== PDF 전자 북마크 (level, page, title) ===")
for level, title, page in doc.get_toc():
    print(f"  L{level} p{page:>2}: {title}")

산업별 항목이 **깊이 3(L3)**에 있습니다. 이번에는 반도체 섹션 첫 페이지를 **일반 변환기(pymupdf4llm 기본 설정)로만** 읽어 봅니다.

In [ ]:
import pymupdf4llm

semi_page = next(p for lvl, t, p in doc.get_toc() if lvl == 3 and "반도체" in t)
raw_md = pymupdf4llm.to_markdown(doc, pages=[semi_page - 1])

print(f"=== 일반 변환 결과 (p{semi_page}) ===")
print(raw_md[:500])
print("\n📌 헤더로 인식된 줄:", [ln for ln in raw_md.splitlines() if ln.startswith("#")])

산업명 `반도체`와 절 제목 `최근 동향`은 헤더가 되지 못했고, 대신 장식 문구가 헤더로 잡혔습니다. **에러도 경고도 없습니다.** 파서는 조용히 틀립니다.
이제 같은 PDF를 정규화 함수로 변환합니다.

In [ ]:
from rag.preprocessing import pdf_to_markdown, BOK_PDF_OPTIONS

bok_md, bok_meta = pdf_to_markdown(bok_pdf, **BOK_PDF_OPTIONS)

# 1. 정규화 후 생성된 마크다운 계층 구조 확인 (H1: 산업, H2: 동향 및 전망)
bok_headers = [ln for ln in bok_md.splitlines() if ln.startswith(("# ", "## ", "### "))]

print(f"📌 {os.path.basename(bok_pdf)}의 마크다운 계층 구조 (정규화 후):")
for h in bok_headers[:9]:
    print(" ", h)
    
print(f"  ... (총 {len(bok_meta['sections'])}개 주력산업에 걸쳐 {len(bok_headers)}개 표준 헤더로 정규화 완료)")

In [ ]:
# Part 3에서 동일한 enricher로 테스트하기 위해 정규화된 Markdown을 저장합니다
os.makedirs("artifacts/normalized", exist_ok=True)
bok_norm_path = f"artifacts/normalized/{os.path.splitext(os.path.basename(bok_pdf))[0]}.md"

with open(bok_norm_path, "w", encoding="utf-8") as f:
    f.write(bok_md)
    
print("\n💾 정규화된 파일 저장 완료:", bok_norm_path)

💡 규격대로 **H1 = 산업**, **H2 = 최근 동향 / 향후 전망**이 만들어졌습니다.
사내 규정(`규정명` ➜ `장` ➜ `조항`)과 한국은행 보고서(`산업` ➜ `최근 동향 / 향후 전망`) 모두 **동일한 표준 마크다운 계층 구조(H1~H2)**를 갖추게 되었습니다!
(`##### …`는 H5라서 청킹 분할 기준(H1~H4)에 쓰이지 않고 본문으로 취급됩니다.)

### 🔍 검수: 정규화가 잘 됐는지 숫자로 확인합니다

앞에서 본 것처럼 파서는 조용히 틀리므로, 몇 페이지를 눈으로 훑어보는 것만으로는 부족합니다. 실무에서는 두 가지를 숫자로 확인합니다.

| 검수 항목 | 확인하는 것 | 구현 위치 |
|:---|:---|:---|
| **텍스트 보존율** | 원문 텍스트가 변환 결과에 빠짐없이 남았는가 | [`rag/preprocessing/quality.py` · `measure_text_coverage()` L8–32](../rag/preprocessing/quality.py) |
| **헤더 분포** | 규격에 없는 헤더(표지 제목, 각주 등)가 생기지 않았는가 | 청크의 h1·h2 값 집계 |

이 보고서의 경우 기본 변환은 본문의 약 10%를 잃지만, 정규화 후에는 **같은 설정으로 4개 분기 모두** 99% 이상을 보존합니다. Mission 02에서 직접 측정해 봅니다.

정규화로도 남는 한계는 **알고 쓰면** 됩니다.

| 한계 | 예 | 대안 |
|:---|:---|:---|
| **차트** | 그래프 속 수치는 텍스트가 아님 | VLM 캡셔닝, 원천 통계 데이터 별도 적재 |
| **표** | 셀이 흩어지거나 값이 빠질 수 있음 | 레이아웃 분석 모델(Docling, Upstage Document Parse 등) |
| **스캔·북마크 없는 문서** | 기댈 구조 정보가 없음 | 레이아웃 분석 모델, VLM 기반 파싱으로 **정규화기만 교체** |

어떤 대안을 쓰든 바뀌는 것은 **정규화기 하나**이고, 출력이 같은 규격이면 이후 파이프라인은 그대로입니다.

---

### 🤖 같은 유형의 문서는 AI 코딩 에이전트와 함께 정규화합니다

"문서마다 파서를 만들어야 한다면 엔지니어링 비용이 너무 크지 않나?" 실제 역할은 이렇게 나뉩니다.

```mermaid
flowchart LR
    subgraph H["👤 사람 (한 번)"]
        S["표준 Markdown 규격<br/>+ 검수 기준"]
    end
    subgraph AI["🤖 AI 코딩 에이전트 (문서 계열마다)"]
        D1["① 구조 진단<br/>북마크 · 반복 소제목 · 노이즈"] --> D2["② 정규화 설정 작성"] --> D3["③ 검수<br/>보존율 · 헤더 분포"]
        D3 -->|"기준 미달"| D2
    end
    subgraph P["♻️ 공용 파이프라인 (수정 없음)"]
        X["청킹 · 문맥 헤더 → 인덱스 → 검색 → 에이전트 도구"]
    end
    S --> D1
    D3 -->|"통과"| X
```

- **사람**은 목표 규격(무엇을 H1·H2로 삼고 어떤 메타데이터를 붙일지)과 검수 기준(예: 보존율 99% 이상, 규격 밖 헤더 0개)을 정합니다.
- **AI 코딩 에이전트**는 문서를 진단해 정규화 설정을 작성하고, 검수 기준을 통과할 때까지 스스로 고칩니다. 문서마다 다른 세부 처리는 이 단계에서 해결됩니다.
- 같은 양식으로 계속 나오는 문서(분기 보고서, 사내 규정집 등)는 **설정 하나로 계속** 처리됩니다.
- 규격을 통과한 문서는 모두 **같은 파이프라인**을 탑니다.

결국 **탈맥락화와 포맷 파편화를 전처리 단계에서 해결**하는 것입니다. 포맷 파편화는 규격 통일로, 탈맥락화는 규격이 만든 계층을 다음 Part의 문맥 헤더로 해결합니다.

<details>
<summary>📋 실습용 프롬프트: 새 양식의 문서 정규화 설정 만들기 (3단계)</summary>

**1단계: 구조 진단**
```text
'{pdf_path}' PDF를 우리 표준 Markdown 규격(H1 = 최상위 주제, H2~H4 = 장·절·조항, 시점 등은 메타데이터)으로
정규화하려고 해. 이 문서가 가진 구조 정보를 조사해 줘.
1. 전자 북마크(doc.get_toc())가 있는지, 깊이별 항목과 시작 페이지
2. 반복되는 소제목, 러닝 헤더·페이지 번호 같은 노이즈의 표기 패턴
3. pymupdf4llm 기본 변환에서 빠지거나 잘못 헤더가 되는 부분이 있는지
```

**2단계: 설정 작성**
```text
진단 결과로 rag/preprocessing/normalize.py의 pdf_to_markdown()에 넘길 설정 dict를 만들어 줘.
함수 자체는 수정하지 말고, LLM 호출 없는 결정론적 규칙만 사용해 줘.
```

**3단계: 전 파일 검수**
```text
설정을 같은 계열의 모든 파일에 적용해서 파일별로 표를 만들어 줘.
1. measure_text_coverage()로 계산한 텍스트 보존율
2. ContextualHeaderEnricher로 청킹했을 때 h1, h2 값별 청크 수
보존율 99% 미만이거나 규격에 없는 헤더가 생긴 파일이 있으면 원인을 찾아 설정을 고쳐 줘.
```
</details>

---
## 📝 Part 3. 청킹: 청크가 스스로 출처를 말하게 (문맥 헤더 주입)

정규화로 계층이 생겼으니, 이제 그 계층 정보를 **모든 청크에 붙여** 탈맥락화를 해결합니다.

```mermaid
flowchart LR
    subgraph T["표준 Markdown 계층"]
        h1["H1 · 국내 여비 및 교통비 지급 규정"] --> h2["H2 · 제3장 숙박비, 일비 및 식비"] --> h3["H3 · 제15조 일비 및 식비"] --> body["본문 청크"]
    end
    T -->|"헤더 경로를 이어 붙임"| B["Breadcrumb<br/>여비 규정 › 제3장 › 제15조"]
    T -->|"모든 헤더를 모음"| TOC["TOC<br/>문서 전체 목차"]
```

문서는 두 단계로 자르고, 각 청크 앞에 문맥 헤더를 붙입니다.
1. `MarkdownHeaderTextSplitter`로 H1~H4 **논리 단위**로 1차 분할
2. 긴 섹션은 `RecursiveCharacterTextSplitter`로 2차 분할

```text
┌─ page_content (임베딩·BM25 검색 대상) ───────────────────────┐
│ --- Document Context Header ---                             │
│ [문서명: (주)넥스트AI 국내 여비 및 교통비 지급 규정]           │
│ [소관부서: ...]                                              │
│ [문서 전체 목차 (TOC)]: • ... - 제3장 ... * 제15조 ...        │  ← 문서 전체 지도
│ [현재 위치 경로: ... > 제3장 숙박비, 일비 및 식비 지급 기준    │  ← Breadcrumb
│                  > 제15조 (일비 및 식비 정액 지급 기준)]      │
│ -------------------------------                             │
│ 2. 당일 출장의 경우 식비는 1식(12,000원)을 지급하며, ...       │  ← 원본 청크
└──────────────────────────────────────────────────────────────┘
  metadata["raw_content"] = 원본 청크만 따로 보관 (Part 6 윈도우 확장에서 사용)
```

| 헤더 요소| 효과 |
|:---|:---|
| **Breadcrumb** | [`rag/indexing/contextual_headers.py` · `ContextualHeaderEnricher._build_breadcrumb()` L139–146](../rag/indexing/contextual_headers.py) | 청크가 어느 조항인지 스스로 설명 → "여비 규정의 일비 기준" 같은 질문과 유사도 상승 |
| **TOC** | [`rag/indexing/contextual_headers.py` · `ContextualHeaderEnricher._extract_table_of_contents()` L113–124](../rag/indexing/contextual_headers.py) | 문서 전체 구조를 함께 임베딩 → 어떤 문서의 일부인지 구분 |
| **raw_content** | [`rag/indexing/contextual_headers.py` · `ContextualHeaderEnricher.process_file()` L39–111](../rag/indexing/contextual_headers.py) | 헤더 없는 원문을 보관 → 생성 단계에서 헤더 중복 없이 이어 붙이기 |

사내 규정과 한국은행 보고서 두 문서에 동일한 `ContextualHeaderEnricher`를 각각 적용하여, 문맥 헤더가 어떻게 주입되는지 눈으로 직접 대조해 봅니다.

### 3.1 문서 1. 사내 규정 청크의 문맥 헤더 주입 결과

In [ ]:
from rag.indexing import ContextualHeaderEnricher

enricher = ContextualHeaderEnricher(chunk_size=500, chunk_overlap=50)

In [ ]:
# 1. 사내 규정 문서 청킹 및 문맥 헤더 주입
policy_chunks, _ = enricher.process_file(policy_files[0], doc_id=0)

sample_pol = next(c for c in policy_chunks if "제15조" in c.metadata.get("breadcrumb", ""))

print("=== 원본 청크 (raw_content - 출처 상실) ===")
print(sample_pol.metadata["raw_content"][:250])

print("\n=== 문맥 헤더 주입 청크 (page_content - 임베딩 & 검색 대상) ===")
print(sample_pol.page_content[:600])

### 3.2 문서 2. 한국은행 보고서(정규화본) 청크의 문맥 헤더 주입 결과

Part 2에서 북마크 기반으로 정규화하여 저장했던 한국은행 보고서(`bok_norm_path`)에 **동일한 `enricher` 객체**를 그대로 통과시킵니다.

In [ ]:
# 2. 한국은행 보고서(정규화된 Markdown) 청킹 및 문맥 헤더 주입
bok_chunks, _ = enricher.process_file(bok_norm_path, doc_id=1)

sample_bok = next(c for c in bok_chunks if "최근 동향" in c.metadata.get("breadcrumb", ""))

print("=== 원본 청크 (raw_content - 수치만 고립) ===")
print(sample_bok.metadata["raw_content"][:250])

print("\n=== 문맥 헤더 주입 청크 (page_content - 임베딩 & 검색 대상) ===")
print(sample_bok.page_content)

💡 **인사이트:**

1. **사내 규정 청크**: 조항 번호(`제15조`)와 지급 기준 표만 덩그러니 분할되어 출처를 잃을 뻔했던 청크가, 상단에 `[문서명]`과 `[현재 위치: ... > 제3장 ... > 제15조 ...]` 헤더를 달아 **"내가 어디 나와있는 규정인지" 맥락에 포함시킵니다.**

2. **한국은행 보고서 청크**: "수출 41.4% 증가"라는 숫자가 8대 산업 중 어디에 해당하는지 알 수 없었던 청크가, 상단에 `[문서명]`과 `[현재 위치: ... > 반도체 > 최근 동향 및 주요 실적]` 헤더가 주입되어 **"반도체의 3분기 실적"이라는 전역 맥락을 포함합니다**

3. **공용 파이프라인의 완성**: **`ContextualHeaderEnricher`의 코드는 수정하지 않습니다.**
   문서들의 포맷 파편화를 해결하고 **표준 Markdown(H1~H4)**으로 규격을 맞췄기 때문에, 사내 규정이든 복잡한 PDF 보고서이든 **동일한 청킹 파이프라인으로 일원화**할 수 있습니다.

> [!NOTE]
> 문맥 헤더에 **전체 목차(TOC)** 를 포함할지(`include_toc`)는 문서 특성과 검색 질문 유형에 따른 설계 선택입니다. 목차를 넣으면 문서의 전체 맥락이 반영되는 장점이 있지만, 같은 문서의 모든 청크가 유사한 목차를 공유하게 되어 단일 조항 번호 검색 시 분별력이 다소 낮아질 수 있습니다.

---
## 🔗 Part 4. 하나의 함수로 통합 인덱싱 + 분기 메타데이터

[`rag/indexing/pipeline.py` · `index_documents()` L19–59](../rag/indexing/pipeline.py)는 **확장자를 보고 정규화기를 고른 뒤([`rag/preprocessing/normalize.py` · `to_markdown()` L83–91](../rag/preprocessing/normalize.py)), 같은 청킹기로** 처리합니다.
Markdown 규정 3개와 PDF 보고서 1개를 **한 번의 호출**로 인덱싱해 봅니다.

### 분기 메타데이터 태깅

한국은행 보고서는 4개 분기가 같은 목차와 문장 구조를 가집니다. 문맥 헤더(산업명)만으로는 **분기**를 확실히 구분하기 어렵습니다.
그래서 파일명(`2024_3사분기_...`)에서 연도와 분기를 뽑아 **청크 메타데이터**로 붙입니다 ([`rag/preprocessing/bok.py` · `bok_metadata()` L28–34](../rag/preprocessing/bok.py)).

| 구분 | 문맥 헤더 (Soft) | 메타데이터 필터 (Hard) |
|:---|:---|:---|
| **어디에 들어가나** | 임베딩되는 텍스트 | 벡터 DB의 메타데이터 필드 (`quarter: 3`) |
| **동작** | 유사도를 높여 관련 청크가 위로 오게 함 | 조건에 맞지 않는 청크를 **아예 제외** |
| **강점** | 질문에 조건이 없어도 맥락으로 찾아감 | 다른 분기 문서가 섞일 가능성 0 |
| **약점** | 본문이 비슷하면 다른 분기 청크가 섞임 | 조건이 생략되거나 여러 분기 비교 질문이면 과도하게 차단 |

> [!TIP]
> 둘 중 하나를 고르는 것이 아니라 **둘 다** 붙입니다. 메타데이터는 인덱싱 때 붙여 두기만 하면 비용이 없고,
> **필터를 언제 쓸지는 에이전트가 정하게** 할 수 있습니다. 4교시에서 `search_bok_reports` 도구에 `quarter` 인자를 열어 이 메타데이터를 활용합니다.

In [ ]:
from collections import Counter
from rag.indexing import index_documents
from rag.preprocessing import bok_metadata

mixed_paths = policy_files + [bok_pdf]
chunks, lookup = index_documents(
    mixed_paths,
    ContextualHeaderEnricher(chunk_size=500, chunk_overlap=50),
    pdf_options=BOK_PDF_OPTIONS,
    extra_metadata=lambda path, meta: bok_metadata(path, meta) if path.endswith(".pdf") else {},
)

print(f"✅ 총 {len(chunks)}개 청크")
for src, n in Counter(c.metadata["source_file"] for c in chunks).items():
    print(f"  • {src}: {n}개")

bok_chunk = next(c for c in chunks if c.metadata.get("quarter") and "반도체" in c.metadata["breadcrumb"])
print("\n=== PDF에서 나온 청크도 같은 형식의 헤더를 가집니다 ===")
print(bok_chunk.page_content.split("-------------------------------")[0][-400:])
print("=== 청크 메타데이터 (분기 태깅) ===")
print({k: bok_chunk.metadata[k] for k in ("year", "quarter", "quarter_name", "breadcrumb")})

---
## 🔎 Part 5. 검색: 의미와 키워드를 함께 (사내 약어 사전 + BM25 + Dense 하이브리드)

### 왜 벡터 검색만으로는 부족한가

| 질문 | Dense (의미) | BM25 (키워드) | 이유 |
|:---|:-:|:-:|:---|
| "출장 가면 밥값은 얼마 줘?" | ✅ | ❌ | 문서에는 `식비`라고 적혀 있음. 의미는 같고 단어는 다름 |
| "제15조 내용 알려줘" | △ | ✅ | 조항 번호는 의미가 아니라 **정확히 같은 글자**가 중요 |
| "DLP 승인 절차" | △ | ✅ | 약어·고유명사는 임베딩 공간에서 흐릿함 |
| "클운팀 출장 결재" | △ | △ | 문서에는 `클라우드운영팀`. **약어 사전**으로 먼저 바꿔야 둘 다 찾음 |

### 검색 파이프라인

```mermaid
flowchart LR
    Q["질문<br/>클운팀 출장 결재는?"] --> D["① 사내 약어 사전<br/>클운팀 → 클라우드운영팀"]
    D --> S["② BM25<br/>키워드 정확 일치"]
    D --> V["② Dense<br/>의미 유사도"]
    S --> F["③ RRF 융합<br/>순위만으로 합치기"]
    V --> F
    F --> K["Top-K"] --> W["④ 윈도우 확장<br/>(Part 6)"]
```

| 단계 | 하는 일 | 구현 위치 |
|:---:|:---|:---|
| ① | 정규식 사전으로 약어·은어를 정식 명칭으로 치환 (LLM 호출 없음) | [`rag/retrieval/domain_dictionary.py` · `DomainDictionary` L12–65](../rag/retrieval/domain_dictionary.py) |
| ② | BM25와 Dense를 각각 검색 | [`rag/retrieval/hybrid_retriever.py` · `HybridRetriever.search()` L55–84](../rag/retrieval/hybrid_retriever.py) |
| ③ | RRF로 두 결과를 하나의 순위로 융합 | [`rag/retrieval/hybrid_retriever.py` · `HybridRetriever._reciprocal_rank_fusion()` L96–119](../rag/retrieval/hybrid_retriever.py) |

### RRF (Reciprocal Rank Fusion): 왜 점수가 아니라 순위로 합치나

Dense 점수(코사인 유사도 0~1)와 BM25 점수(0~30 이상)는 **단위가 달라 더할 수 없습니다.** RRF는 점수를 버리고 **순위**만 씁니다.

$$\text{RRF}(d) = \sum_{r \in \{\text{BM25},\ \text{Dense}\}} \frac{1}{k + \text{rank}_r(d)} \qquad (k = 60)$$

| 청크 | BM25 순위 | Dense 순위 | RRF 점수 | 최종 |
|:---|:-:|:-:|:---|:-:|
| A | 1 | 2 | 1/61 + 1/62 = **0.0325** | 🥇 |
| C | 3 | 3 | 1/63 + 1/63 = **0.0317** | 🥈 |
| B | — | 1 | 1/61 = **0.0164** | 🥉 |

**양쪽에서 고르게 상위인 청크**가 한쪽에서만 1위인 청크보다 앞섭니다. 두 검색기가 서로의 실수를 보완하는 구조입니다.

> [!TIP]
> **한국어 BM25 주의**: LangChain `BM25Retriever`는 기본적으로 **공백**으로 토큰을 나눕니다.
> 한국어는 조사가 붙고(`출장비는`, `출장비를`), PDF에서 띄어쓰기가 사라지기도 해서(Part 2) 키워드가 일치하지 않을 수 있습니다.
> 운영 환경에서는 `preprocess_func`에 형태소 분석기(예: Kiwi)를 연결해 명사 단위로 토큰화하는 것을 권장합니다.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import BM25Retriever
from rag.retrieval import HybridRetriever

# 노트북 실습용 인메모리 인덱스 (app/database의 운영 DB는 건드리지 않습니다)
dense_store = Chroma.from_documents(
    chunks, OpenAIEmbeddings(model="text-embedding-3-large"), collection_name="nb2_demo"
)
bm25 = BM25Retriever.from_documents(chunks, k=5)

hybrid = HybridRetriever(
    vectorstore=dense_store,
    bm25_retriever=bm25,
    chunk_lookup_map=lookup,
    use_hyde=False,          # HyDE는 LLM 호출이 필요하므로 실습에서는 끔
    use_window_expansion=True,
    window_size=1,
)
print("✅ 하이브리드 검색기 준비 완료")
print("약어 사전 치환 예:", hybrid.domain_dict.replace_synonyms("클운팀 직원 출장 결재는 누가 최종 승인해?"))

In [ ]:
query = "클운팀 직원 출장 결재는 누가 최종 승인해?"

print("🔹 Dense 단독 검색 Top-2")
for d in dense_store.similarity_search(query, k=2):
    print("   ", d.metadata["breadcrumb"])

print("\n🔸 하이브리드 검색 Top-2 (약어 사전 + BM25 + RRF + 윈도우 확장)")
for d in hybrid.search(query, top_k=2):
    print("   ", d.metadata["breadcrumb"], f"(청크 {d.metadata.get('window_start_chunk')}~{d.metadata.get('window_end_chunk')} 결합)")

💡 이 질문은 `클운팀`이라는 약어가 제21조 본문에 그대로 있어 Dense 검색도 1위를 맞힙니다.
차이는 **무엇을 가져오느냐**에 있습니다. 하이브리드 검색기는 찾은 청크 하나가 아니라 **앞뒤 청크를 이어 붙인 덩어리**를 돌려줍니다. 이것이 다음 파트의 윈도우 확장입니다.

---
## 🪟 Part 6. 생성용 컨텍스트: 작게 찾고 크게 읽기 (Window expansion)

청크 크기에는 딜레마가 있습니다.

| 청크 크기 | 검색 정확도 | 생성 품질 |
|:---|:---|:---|
| **작게** (300~500자) | ✅ 주제가 하나라 유사도가 선명함 | ❌ 표가 중간에 잘리고, 단서 조항(②, ③항)이 다음 청크로 넘어감 |
| **크게** (2,000자 이상) | ❌ 여러 주제가 섞여 유사도가 흐려짐 | ✅ 맥락이 온전함 |

윈도우 확장은 **검색은 작은 청크로, LLM에 넘길 때는 앞뒤 청크까지** 붙여서 두 장점을 모두 얻습니다.

```mermaid
flowchart TD
    subgraph S1["1단계: 작은 청크로 검색"]
        direction LR
        c1["청크 13"] ~~~ c2["청크 14"] ~~~ c3["🎯 청크 15<br/>검색 Hit"] ~~~ c4["청크 16"] ~~~ c5["청크 17"]
    end
    subgraph S2["2단계: 룩업 맵으로 앞뒤 청크 조회 (window_size=1)"]
        direction LR
        e2["청크 14<br/>(-1)"] ~~~ e3["🎯 청크 15"] ~~~ e4["청크 16<br/>(+1)"]
    end
    subgraph S3["3단계: 하나의 컨텍스트로 결합"]
        m["중심 청크의 문맥 헤더 1회<br/>+ 청크 14·15·16의 raw_content"]
    end
    S1 --> S2 --> S3
```

| 구성 요소 | 역할 |
|:---|:---|
| 룩업 맵 `{(doc_id, chunk_id): 청크}` | 인덱싱 때 `index_documents()`가 함께 만들어 둠. 앞뒤 청크를 DB 재검색 없이 즉시 조회 |
| `raw_content` | 이웃 청크는 헤더 없이 원문만 붙여 **헤더가 3번 반복되지 않게** 함 |
| `window_size` | 앞뒤 몇 개까지 붙일지. 키울수록 맥락은 늘고 토큰 비용도 늘어남 |

구현 위치: [`rag/retrieval/context_window_expander.py` · `ChunkWindowExpander.expand_documents()` L25–79](../rag/retrieval/context_window_expander.py)

In [ ]:
# 윈도우 확장으로 결합된 컨텍스트 확인 (중심 청크 + 앞뒤 청크)
expanded = hybrid.search("당일 출장 일비 식비 지급 기준", top_k=1)[0]
print(f"📍 {expanded.metadata['breadcrumb']} | 청크 {expanded.metadata['window_start_chunk']}~{expanded.metadata['window_end_chunk']} 결합\n")
print(expanded.page_content.split("-------------------------------")[-1][:1200])

💡 중심 청크(감액 기준) 앞뒤로 이어지는 조항이 함께 들어와, 단서 조항이나 표가 청크 경계에서 잘려도 LLM은 전체를 보고 답할 수 있습니다.

---
## 🕸️ Part 7. 지식 그래프: 다중 홉 관계 질문 (Knowledge Graph & Multi-Hop)

### ❓ [실무 문제 제기] Vector RAG(순수 벡터 검색)의 다중 홉 한계

> *"클라우드운영팀 김철수 수석이 총괄하는 프로젝트의 예산을 최종 승인하는 본부장은 누구인가?"*

이 질문에 올바르게 답하기 위해서는 다음과 같은 **다단계 인과 관계(Multi-Hop)**를 거쳐야 합니다:

1. **Hop 1 (담당 프로젝트)**: `[김철수 수석]` ──(총괄 PM)──▶ `[P-01: 클라우드 네이티브 마이그레이션 (예산 3.5억)]`
2. **Hop 1 (소속 부서)**: `[김철수 수석]` ──(팀장으로 재직)──▶ `[클라우드운영팀]`
3. **Hop 2 (상위 조직)**: `[클라우드운영팀]` ──(산하 조직 포함)──▶ `[클라우드사업본부]`
4. **Hop 2 (예산 승인권자)**: `[P-01 (예산 3.5억)]`은 전결 위임 규정(5천만~5억원)에 따라 ──▶ `[클라우드사업본부장 박영희 전무]`

```mermaid
graph LR
    A["김철수 수석"] -->|팀장으로 재직| B["클라우드운영팀"]
    A -->|총괄 PM| C["P-01: 마이그레이션 (3.5억)"]
    B -->|산하 조직 포함| D["클라우드사업본부"]
    C -->|예산 승인권자| E["★ 박영희 본부장 (최종 승인권자)"]
    D -->|본부장으로 재직| E

    style A fill:#e1f5fe,stroke:#0288d1,stroke-width:2px
    style C fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style E fill:#e8f5e9,stroke:#388e3c,stroke-width:3px
```

### ❌ 왜 벡터 검색만으로는 이 질문을 풀 수 없을까요?
- 사내 정보는 **조직도 청크**(`김철수 팀장`), **프로젝트 개요 청크**(`P-01 예산 3.5억`), **전결 위임 규정 청크**(`5천만~5억은 본부장 승인`), **본부장 정보 청크**(`클라우드사업본부장은 박영희`)로 각각 쪼개져 있습니다.
- 어떤 단일 청크도 *"김철수의 승인자는 박영희"*라는 텍스트를 직접 담고 있지 않기 때문에, 코사인 유사도 기반 벡터 검색은 중간 링크가 끊어져 **환각을 일으키거나 검색에 실패**합니다.

### 🕸️ 지식 그래프(Knowledge Graph)의 해법: 경량 도메인 온톨로지 주입

문장을 `(주어, 관계, 목적어)` 형태의 **트리플렛(Triplet)**으로 변환하여 노드와 엣지로 명시 저장합니다.

> 💡 **LLM 시대의 도메인 온톨로지(Domain Ontology) 주입**:
> LLM에게 제약 없이 관계를 뽑게 하면 `이끈다`, `팀장이다`, `수장이다`처럼 관계 어휘가 파편화되어 그래프 검색이 불가능해집니다.  
> 현대 엔지니어링에서는 복잡한 학술 온톨로지 대신, **Pydantic 스키마와 프롬프트로 핵심 개체(인물·부서·프로젝트)와 표준 관계 어휘(소속본부·직속팀장·총괄PM·승인권자)를 명확히 제약(Constraint)**하여 주입합니다.

In [ ]:
with open("data/graph/org_relationships.txt", "r", encoding="utf-8") as f:
    org_text = f.read()
print(org_text[:700])

In [ ]:
from rag.graph.triplet_extractor import extract_triplets

kg = extract_triplets(org_text, llm=llm)   # Pydantic 구조화 출력으로 트리플렛 추출
print(f"✅ 트리플렛 {len(kg.triplets)}개 추출\n")
for t in kg.triplets[:12]:
    print(f"  [{t.subject}] ──({t.relation})──▶ [{t.object}]")

In [ ]:
from rag.graph.networkx_graph import build_graph_from_triplets, visualize_pyvis, multi_hop_search
from IPython.display import IFrame

G = build_graph_from_triplets(kg.triplets)
print(f"노드 {G.number_of_nodes()}개, 엣지 {G.number_of_edges()}개")

os.makedirs("artifacts", exist_ok=True)
html_path = visualize_pyvis(G, output_html="artifacts/knowledge_graph.html")
print(f"🌐 인터랙티브 그래프 저장: {html_path} (브라우저로 열면 노드를 드래그할 수 있습니다)")

# IFrame 경로는 노트북 파일 위치(notebooks/) 기준이므로 상위 폴더로 올라갑니다.
IFrame("../" + html_path, width="100%", height=520)

In [ ]:
for hops in (1, 2):
    edges, visited = multi_hop_search(G, seed_entities=["김철수"], max_hops=hops)
    print(f"\n🔍 {hops}-Hop 탐색: 방문 노드 {len(visited)}개, 관계 {len(edges)}개")
    for u, v, d in edges[:8]:
        print(f"   [{u}] ──({d['relation']})──▶ [{v}]")

💡 **1-Hop vs 2-Hop 탐색 대조와 엔지니어링 인사이트:**

1. **1-Hop 탐색 (직접 연결)**: 시작점인 `김철수`에서 바로 뻗어나간 노드(`클라우드운영팀`, `P-01 프로젝트`)까지만 방문하므로, 최종 결재권자인 `박영희 본부장`에게 도달하지 못합니다.
2. **2-Hop 탐색 (다중 홉 추론)**: `클라우드운영팀`의 상위 본부인 `클라우드사업본부`와 협업 부서, 그리고 P-01 프로젝트의 예산 승인권자인 `박영희 본부장`까지 엣지를 타고 확장되어 **다중 홉 관계가 온전히 복원**됩니다!
3. **에이전트 도구 설계 원칙**: 홉 수가 커지면 먼 관계까지 찾을 수 있지만, 방문 노드가 기하급수적으로 늘어 LLM 컨텍스트가 오염될 수 있습니다.
   따라서 1교시의 `search_graph_relations` 도구처럼 **`max_hops`를 인자로 노출**하여, 에이전트가 질문의 복잡도에 따라 탐색 깊이(1-Hop vs 2-Hop)를 스스로 판단해 호출하도록 설계합니다.

> [!NOTE]
> **실무 데이터와 엔티티 정규화(Entity Resolution)**:
> 1. 본 실습 데이터는 그래프 순회 알고리즘과 GraphRAG의 메커니즘을 명확히 체감할 수 있도록 정제된 골든 셋입니다.
> 2. 실제 비정형 텍스트에서는 문맥에 따른 표기 불일치(`김철수 수석` / `김철수 팀장`)를 하나의 표준 노드로 묶어주는 **엔티티 정규화(동의어 통합)** 파이프라인이 선행됩니다.

---
## 🌐 Part 8. Microsoft GraphRAG: 전사 거시 질문 (Global Sensemaking)

### ❓ 직접 구축한 지식 그래프의 한계와 GraphRAG의 필요성
- Part 7의 경량 그래프(DiGraph)는 *"김철수의 결재선은?"* 처럼 **특정 엔티티가 명시된 미시적(Local) 관계 추적**에는 최적입니다.
- 하지만 *"전사 프로젝트들의 공통 목표와 리스크는?"* 같은 질문은 **특정 시작 엔티티가 없는 전사 거시 질문(Global Sensemaking)**입니다.
  어떤 노드에서 탐색을 시작해야 할지 알 수 없으므로, 전통적인 지식 그래프 탐색으로는 답을 찾을 수 없습니다!

Microsoft Research의 **GraphRAG**는 인덱싱 단계에서 그래프를 **Leiden 계층 군집화(Hierarchical Clustering)**로 묶고, 각 커뮤니티마다 **LLM 심층 요약 리포트(Community Report)**를 미리 생성해 둠으로써 이 문제를 완벽하게 해결합니다.

---

### 🏗️ Microsoft GraphRAG 10-Stage 구축 파이프라인

```mermaid
flowchart TD
    A["사내 원천 텍스트 (조직/프로젝트)"] --> B["① Text Chunking (텍스트 청킹)"]
    B --> C["② LLM Entity & Relationship Extraction (엔티티·관계 추출)"]
    C --> D["③ Graph Merge & Finalize (동의어 통합 & 그래프 구축)"]
    D --> E["④ Leiden Hierarchical Clustering (계층적 커뮤니티 군집화)"]
    E -->|Level 0, 1, 2| F["⑤ LLM Community Report Generation (커뮤니티 요약 리포트)"]
    F --> G["⑥ Vector Embedding (리포트 & 엔티티 벡터화)"]
    G --> H["📦 사전 구축 산출물 (app/database/graphrag/output/)"]
```

### 📂 저장된 GraphRAG 산출물 구조 (`app/database/graphrag/output/`)
- `entities.parquet`: 추출된 엔티티(인물, 팀, 프로젝트) 노드 메타데이터
- `relationships.parquet`: 엔티티 간 방향성 관계 엣지 및 가중치
- `communities.parquet`: Leiden 알고리즘으로 계층화된 커뮤니티 군집 정보
- `community_reports.parquet`: **[핵심 소스] 12개 커뮤니티별 LLM 심층 요약 리포트 (Global Search가 0ms로 거시 질문에 답하는 원천)**
- `lancedb/`: 엔티티 및 텍스트 청크 벡터 인덱스 (Local Search의 소스)

---

### 🧠 Global Search vs Local Search 메커니즘 대조

```mermaid
flowchart TD
    subgraph Global_Search ["1. Global Search: 거시 질문 Map-Reduce"]
        direction TB
        Q1["질문: 전사 프로젝트 공통 목표/리스크"] --> C1["전체 커뮤니티 리포트 로드"]
        C1 --> M1["Map: 커뮤니티별 핵심 포인트 병렬 추출"]
        M1 --> R1["Reduce: 전사 종합 리포트 생성 및 출처 인용"]
    end

    subgraph Local_Search ["2. Local Search: 미시 질문 서브그래프 융합"]
        direction TB
        Q2["질문: 김철수 수석의 역할/보고 라인"] --> V2["LanceDB 엔티티 벡터 매칭"]
        V2 --> E2["1~2 Hop 서브그래프 Triplet 확장"]
        E2 --> F2["소속 커뮤니티 맥락 + 원본 텍스트 청크 융합"]
        F2 --> A2["정밀 결재선/보고 라인 답변 도출"]
    end
```

| 검색 방식 | 동작 메커니즘 | 적합한 질문 예시 |
|:---|:---|:---|
| **Global Search** | 모든 커뮤니티 리포트를 **Map-Reduce**로 종합 | 전사 트렌드, 공통 리스크, 전체 종합 요약 |
| **Local Search** | 질문 속 엔티티 주변 **1~2 Hop 서브그래프 + 원문 청크** 결합 | 특정 인물, 부서, 프로젝트의 세부 사실 정보 |

> 💡 인덱싱은 LLM 호출이 많아 수 분이 걸리므로, 실습에서는 사전 빌드된 산출물(`app/database/graphrag/output/`)을 활용하여 즉시 고성능 추론을 실습합니다.

In [ ]:
import pandas as pd

out_dir = "app/database/graphrag/output"
entities = pd.read_parquet(f"{out_dir}/entities.parquet")
reports = pd.read_parquet(f"{out_dir}/community_reports.parquet")
print(f"엔티티 {len(entities)}개, 커뮤니티 리포트 {len(reports)}개\n")
for _, row in reports.head(3).iterrows():
    print(f"[커뮤니티 {row['community']}] {row['title']}\n   {row['summary'][:150]}...\n")

In [ ]:
from rag.graph.graphrag_tool import run_graphrag_query

print("🌐 [Global] 전사 프로젝트들의 공통 목표와 리스크는?\n")
print(run_graphrag_query("전사 프로젝트들의 공통 목표와 핵심 리스크를 요약해줘.", method="global",
                         root_dir="app/database/graphrag")[:1200])

In [ ]:
print("🎯 [Local] 김철수 수석의 역할과 보고 라인은?\n")
print(run_graphrag_query("클라우드운영팀 김철수 수석의 역할과 보고 라인은 어떻게 되나요?", method="local",
                         root_dir="app/database/graphrag")[:1200])

💡 **Global Search vs Local Search 실행 결과 인사이트:**

1. **Global Search (전사 거시 질문)**:
   - 개별 텍스트 청크를 일일이 검색하지 않고도, Leiden 군집 기반의 커뮤니티 리포트들을 병렬 Map-Reduce하여 **P-01(마이그레이션), P-02(Agentic RAG), P-03(거버넌스)의 상호 연계 구조와 인력 의존성 리스크**를 단번에 거시적으로 조망해 냅니다.
2. **Local Search (특정 엔티티 질문)**:
   - `김철수 수석`이라는 엔티티를 중심으로 주변 1~2 Hop 지식 그래프 관계와 소속 커뮤니티 맥락을 결합하여, **팀장 직무, PM 역할, 상위 보고 라인(박영희 본부장)**을 누락 없이 정확하게 서술합니다.

---
## 🎓 엔터프라이즈 3대 인덱싱 기술 종합 비교 (Key Takeaways)

| 구분 | Naive Vector RAG | 기초 Graph RAG (Track 1) | Microsoft GraphRAG (Track 2) |
| :--- | :--- | :--- | :--- |
| **핵심 기술** | 문맥 헤더 청킹 + Dense/BM25 (Part 3, 5) | Pydantic Triplet + NetworkX DiGraph (Part 7) | Leiden 계층 군집화 + Map-Reduce 리포트 (Part 8) |
| **적합한 질문** | 단일 단락 사실 질의, 규정/조항 검색 | 2~3단계 다중 홉(Multi-hop) 조직/결재선 | 전사적 거시 질문 (트렌드, 리스크, 공통 목표) |
| **장점** | 구현 속도 빠름, 조항 번호 정확 일치 | 가볍고 명확한 인과/소속 경로 추적 | 흩어진 데이터 전체를 아우르는 통찰(Sensemaking) |
| **에이전트 도구** | `search_policy_documents`, `search_bok_reports` | `search_graph_relations` | `query_enterprise_graphrag` (Global/Local) |

> **🌟 실무 완성형 아키텍처**:
> 엔터프라이즈 Agentic RAG 플랫폼에서는 "어떤 인덱스가 제일 좋은가?"를 다투는 것이 아닙니다.

> 데이터와 질의 유형에 따라 AI 에이전트가 검색 컨텍스트로 다루기 좋은 형태로 변환하는 것 입니다.

> 그런 다음 **LangGraph 에이전트가 사용자의 질문 의도를 분석하여 최적의 도구를 자율적으로 선택(Routing)하여 다단계 추론을 수행**하도록 구축하는 것이 엔터프라이즈 RAG의 표준 정석입니다!

---
## 🏁 Part 9. Mission 02

### 🎯 Mission 02. 데이터베이스 빌드 및 검증

`missions/02_missions.md`의 미션을 수행하세요.
1. 이 노트북의 파이프라인을 그대로 쓰는 `app/mcp/build_database.py`로 운영 DB를 다시 빌드합니다.
2. `app/mcp/test_database.py`로 4대 DB 검색을 검증합니다.
3. 새 문서를 추가해 인덱싱하고, 1교시의 에이전트가 새 문서로 답하는지 확인합니다.